# YOLOv9 Fire & Smoke Detection (Ultralytics)

Run top to bottom. Edit the **Config** cell first.

Notes for this task:
- Keep **color** images (color is a key signal for fire).
- Hue shift is off and vertical flip is off (flames rise, orange should stay orange).
- Include **hard negatives** (sunsets, lamps, fog, steam) with empty label files.
- If your images come from video, **split by video**, not by frame.

## 1. Install

In [ ]:
# Install PyTorch first (https://pytorch.org, match your CUDA version), then:
%pip install -q ultralytics

## 2. Check GPU

In [ ]:
import os, torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected, training will be very slow.")
print("Working directory:", os.getcwd())

## 3. (Colab only) Mount Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## 4. Config
Use **absolute paths** if unsure where the notebook runs from.

In [ ]:
from pathlib import Path

# --- Data / model ---
DATA_YAML = "dataset.yaml"
MODEL     = "yolov9c.pt"     # yolov9t/s/m/c/e.pt

# --- Training ---
EPOCHS    = 10
IMGSZ     = 416              # try 960 / 1280 if distant fire/smoke is missed
BATCH     = 4                # -1 = auto
WORKERS   = 0                # 0 avoids multiprocessing issues in notebooks / Windows
PATIENCE  = 5               # early stopping (0 = off)
PROJECT   = r"your-root-path-file"
RUN_NAME  = "baseline"

# --- Inference (tune these AFTER training, see section 9) ---
CONF = 0.20                  # lower = catch more fires, more false alarms
IOU  = 0.50                  # NMS overlap; lower merges overlapping smoke boxes

assert os.path.exists(DATA_YAML), f"Dataset YAML not found: {os.path.abspath(DATA_YAML)}"

## 5. Dataset sanity check
Counts images, missing/empty labels (background images) and instances per class.
Look for class imbalance and make sure you have background images.

In [ ]:
import yaml
from collections import Counter

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
cfg = yaml.safe_load(open(DATA_YAML))
root = Path(cfg.get("path", "."))
names = cfg["names"] if isinstance(cfg["names"], dict) else dict(enumerate(cfg["names"]))

for split in ["train", "val"]:
    img_dir = root / cfg[split]
    lbl_dir = Path(str(img_dir).replace("images", "labels"))
    imgs = [p for p in img_dir.rglob("*") if p.suffix.lower() in IMG_EXT]
    counts, empty, missing = Counter(), 0, 0
    for p in imgs:
        lf = lbl_dir / (p.stem + ".txt")
        if not lf.exists():
            missing += 1
            continue
        lines = [l.split() for l in lf.read_text().splitlines() if l.strip()]
        if not lines:
            empty += 1
        for l in lines:
            counts[int(l[0])] += 1
    print(f"[{split}] images: {len(imgs)} | empty labels: {empty} | no label file: {missing}")
    for k, v in sorted(counts.items()):
        print(f"    {names.get(k, k)}: {v} boxes")
    if len(imgs) == 0:
        print("    !! No images found, check 'path' and folder names in the YAML")

## 6. Train
Augmentation is explicit here so you can see and change it.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    patience=PATIENCE,
    device=0 if torch.cuda.is_available() else "cpu",
    project=PROJECT,
    name=RUN_NAME,
    pretrained=True,
    plots=True,

    # --- Augmentation (fire/smoke settings) ---
    hsv_h=0.0,        # no hue shift: keeps fire orange/red
    hsv_s=0.7,        # saturation variation
    hsv_v=0.4,        # brightness variation (day / dusk / night)
    fliplr=0.5,       # horizontal flip OK
    # flipud=0.0,       # never flip vertically (flames rise)
    mosaic=1.0,       # helps with varied sizes/backgrounds
    scale=0.5,        # fire/smoke appear at very different sizes
    translate=0.1,
    degrees=0.0,
    mixup=0.0,        # try 0.1 if overfitting
    close_mosaic=10,  # mosaic off for the last N epochs

    # freeze=10,      # freeze first N layers (tiny datasets)
    # cache=True,     # faster epochs if you have enough RAM
)

SAVE_DIR = Path(model.trainer.save_dir)
BEST = SAVE_DIR / "weights" / "best.pt"
print("Run folder:", SAVE_DIR)
print("Best weights:", BEST)

## 7. Validate (per-class results)

In [ ]:
model = YOLO(str(BEST))
metrics = model.val(data=DATA_YAML, imgsz=IMGSZ, conf=0.001, iou=0.6)

print(f"\nOverall  mAP50: {metrics.box.map50:.4f} | mAP50-95: {metrics.box.map:.4f}")
print(f"Overall  precision: {metrics.box.mp:.4f} | recall: {metrics.box.mr:.4f}")
for i, cls_idx in enumerate(metrics.box.ap_class_index):
    p, r, ap50, ap = metrics.box.class_result(i)
    print(f"  {model.names[int(cls_idx)]:<8} P={p:.3f}  R={r:.3f}  mAP50={ap50:.3f}  mAP50-95={ap:.3f}")

## 8. View curves and save result
Check: losses falling, val loss not rising (overfitting), confusion matrix (fire vs smoke vs background).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display


# CONFIG: change things here only
# =====================================================================

# --- Where the results come from ---
RELOAD_FROM_DISK = False     # True if the kernel restarted (rebuilds SAVE_DIR, model, metrics)
RUN_DIR = "model/baseline"   # only used when RELOAD_FROM_DISK = True

# --- Saved plots to display (delete any you don't want) ---
PLOTS_TO_SHOW = [
    "results.png",                       # loss + mAP over epochs
    "confusion_matrix.png",              # raw counts
    "confusion_matrix_normalized.png",   # percentages
    "BoxPR_curve.png",                   # precision vs recall
    "BoxF1_curve.png",                   # F1 vs confidence
    # "BoxP_curve.png",
    # "BoxR_curve.png",
]

# --- Epoch table ---
SHOW_LAST_N_EPOCHS = 10      # None = show all epochs
TABLE_COLUMNS = [
    "epoch", "train/box_loss", "val/box_loss",
    "metrics/precision(B)", "metrics/recall(B)",
    "metrics/mAP50(B)", "metrics/mAP50-95(B)",
    # "val/cls_loss", "val/dfl_loss",
]
BEST_EPOCH_BY = "metrics/mAP50-95(B)"   # or "metrics/mAP50(B)" / "metrics/recall(B)"

# --- Confusion matrix drawn here ---
SHOW_MATRIX = True
MATRIX_NORMALIZE = True      # True = fractions (0-1), False = raw box counts
MATRIX_FIGSIZE = (5, 4)
MATRIX_CMAP = "Blues"        # "Reds", "Greens", "viridis", ...


# Results
# =====================================================================

if RELOAD_FROM_DISK:
    from ultralytics import YOLO
    SAVE_DIR = Path(RUN_DIR)
    model = YOLO(str(SAVE_DIR / "weights" / "best.pt"))
    metrics = model.val(data=DATA_YAML, imgsz=IMGSZ)

# 1) Saved plots
for f in PLOTS_TO_SHOW:
    fp = SAVE_DIR / f
    if fp.exists():
        print(f)
        display(Image(filename=str(fp)))
    else:
        print(f"(not found: {f})")

# 2) Epoch table
df = pd.read_csv(SAVE_DIR / "results.csv")
df.columns = df.columns.str.strip()
cols = [c for c in TABLE_COLUMNS if c in df.columns]
missing = [c for c in TABLE_COLUMNS if c not in df.columns]
if missing:
    print("Columns not in results.csv (skipped):", missing)
    print("Available columns:", df.columns.tolist())
display(df[cols] if SHOW_LAST_N_EPOCHS is None else df[cols].tail(SHOW_LAST_N_EPOCHS))

if BEST_EPOCH_BY in df.columns:
    best = df.loc[df[BEST_EPOCH_BY].idxmax()]
    print(f"Best epoch by {BEST_EPOCH_BY}: {int(best['epoch'])} (value {best[BEST_EPOCH_BY]:.4f})")

# 3) Confusion matrix
if SHOW_MATRIX:
    cm = np.array(metrics.confusion_matrix.matrix)   # rows = predicted, cols = true
    labels = [model.names[i] for i in range(len(model.names))] + ["background"]

    if MATRIX_NORMALIZE:
        data = cm / (cm.sum(axis=0, keepdims=True) + 1e-9)
        vmax, fmt = 1, "{:.2f}"
    else:
        data = cm
        vmax, fmt = cm.max(), "{:.0f}"

    fig, ax = plt.subplots(figsize=MATRIX_FIGSIZE)
    im = ax.imshow(data, cmap=MATRIX_CMAP, vmin=0, vmax=vmax)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_xlabel("True"); ax.set_ylabel("Predicted")
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, fmt.format(data[i, j]), ha="center", va="center",
                    color="white" if data[i, j] > vmax * 0.5 else "black")
    plt.colorbar(im); plt.tight_layout(); plt.show()

## 9. Pick a confidence threshold
The F1 curve shows the confidence with the best precision/recall balance.
For fire, missing a fire is worse than a false alarm, so use a value **somewhat below** the peak.

In [ ]:
import numpy as np

try:
    f1 = np.array(metrics.box.f1_curve)   # (classes, 1000)
    px = np.array(metrics.box.px)         # confidence values 0..1
    mean_f1 = f1.mean(0)
    best_all = px[mean_f1.argmax()]
    print(f"Best overall F1 at conf = {best_all:.2f}  (F1 = {mean_f1.max():.3f})")
    for i, cls_idx in enumerate(metrics.box.ap_class_index):
        print(f"  {model.names[int(cls_idx)]:<8} best F1 at conf = {px[f1[i].argmax()]:.2f}")
    print(f"\nSuggestion: start around conf = {max(best_all - 0.1, 0.05):.2f} for higher recall, then test on real video.")
except Exception as e:
    print("Could not compute from metrics, read the peak off BoxF1_curve.png instead.", e)

## 10. Predict on images
Use fresh photos from real conditions, ideally ones the model never saw.

In [ ]:
preds = model.predict(
    source=r"test-folder-path-images",   # image, folder, or video
    conf=CONF, iou=IOU, imgsz=IMGSZ,
    save=True,                      # saved under runs/detect/predict*
)
for r in preds[:5]:
    print(Path(r.path).name, "->", [(model.names[int(c)], round(float(s), 2)) for c, s in zip(r.boxes.cls, r.boxes.conf)])

## 11. Video with persistence (fewer false alarms)
Alerts only trigger if fire/smoke appears in at least `min_hits` of the last `window` frames,
which filters out single-frame flicker. Set `vid_stride=2` or more to skip frames and save compute.

In [ ]:
import cv2
from collections import deque

def run_video(source, out_path="alert_output.mp4", conf=CONF, iou=IOU,
              window=10, min_hits=5, vid_stride=1):
    cap = cv2.VideoCapture(source)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps / vid_stride, (w, h))
    history = {n: deque(maxlen=window) for n in model.names.values()}
    alert_frames = 0

    stream = model.predict(source=source, conf=conf, iou=iou, imgsz=IMGSZ,
                           stream=True, vid_stride=vid_stride, verbose=False)
    for r in stream:
        present = {model.names[int(c)] for c in r.boxes.cls}
        for name, hist in history.items():
            hist.append(name in present)
        active = [n for n, hv in history.items() if sum(hv) >= min_hits]

        frame = r.plot()
        if active:
            alert_frames += 1
            cv2.putText(frame, "ALERT: " + ", ".join(active), (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 3)
        writer.write(frame)

    writer.release()
    print(f"Saved {out_path} | frames with alert: {alert_frames}")

# run_video("path/to/video.mp4", vid_stride=2)

## 12. Export
`onnx` (general), `engine` (TensorRT / Jetson), `openvino` (Intel CPU), `tflite` (mobile / small devices).
Test the exported model against `best.pt` on a few images.

In [ ]:
exported = model.export(format="onnx", imgsz=IMGSZ)
print("Exported to:", exported)

## 13. Improve with real-world mistakes
Run the model on footage from the real camera (day, night, rain, fog, glare, cooking smoke, steam).
Collect every **false alarm** (add as background images with empty labels) and every **miss**
(label and add them), then retrain. This is usually the biggest improvement.

## Resume an interrupted run
Restart the kernel, run the Install cell, then:

In [ ]:
# from ultralytics import YOLO
# YOLO("runs/firesmoke/baseline/weights/last.pt").train(resume=True)